# Lab 06-07 · Linearity and Superposition

Companion to chapter [06-07](../../chapters/06-07-linearity-and-superposition.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

def run_ngspice(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                vals[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    return vals

ngspice available: True


## Part 1 — Superposition in ngspice

[superposition.cir](superposition.cir) contains each example three times: all sources on, and each source alone with the others set to zero (a 0 V source is a short, a 0 A source an open).

**🔮 Predict:** in Example 06-07.1, which source delivers power, and how much?

In [2]:
if HAVE_NGSPICE:
    ng = run_ngspice("superposition.cir")
    for total, parts in (("ex1all", ("ex1v", "ex1i")), ("ex2all", ("ex2v", "ex2i")), ("p12all", ("p12v", "p12i"))):
        s = sum(ng[p] for p in parts)
        print(f"{total:7s} all sources {ng[total]: .6f} V   sum of single-source runs {s: .6f} V")
    assert abs(ng["ex1all"] - ng["ex1v"] - ng["ex1i"]) < 1e-6
    assert abs(ng["ex2all"] - ng["ex2v"] - ng["ex2i"]) < 1e-5
    assert abs(ng["p12all"] - ng["p12v"] - ng["p12i"]) > 0.5      # the diode circuit does not superpose
else:
    print("ngspice not found; see SETUP.md. Skipping.")

va = 12.0                                  # Example 06-07.1 by hand
print(f"Example 06-07.1: 12 V source current {(12 - va) / 4:.3f} A -> power {12 * (12 - va) / 4:.1f} W; 2 A source power {2 * va:.1f} W")

ex1all  all sources  12.000000 V   sum of single-source runs  12.000000 V
ex2all  all sources  7.333330 V   sum of single-source runs  7.333337 V
p12all  all sources  0.697914 V   sum of single-source runs  1.321690 V
Example 06-07.1: 12 V source current 0.000 A -> power 0.0 W; 2 A source power 24.0 W


The 12 V source delivers **nothing**: node a sits at exactly 12 V, so no current flows through $R_1$. All 24 W come from the 2 A source. The linear examples add up exactly; the diode circuit (Problem 06-07.12) does not.

## Part 2 — Watching linearity fail

Sweep the strength of the second source in the diode circuit and compare the true node voltage with the superposed one.

**🔮 Predict:** for small injected currents, does superposition become accurate again?

In [3]:
IS, VT = 1e-14, 0.025852
def vd(Vs, I):
    return brentq(lambda v: (Vs - v) / 1e3 + I - IS * np.expm1(v / VT), -1, 1, xtol=1e-15)

v5 = vd(5, 0)
I = np.logspace(-6, -2, 41)
true = np.array([vd(5, i) for i in I]) - v5          # change caused by adding I with the 5 V present
superposed = np.array([vd(0, i) for i in I])          # what superposition would add
rd = VT / (IS * np.expm1(v5 / VT) + IS)
small = I * 1 / (1 / 1e3 + 1 / rd)                    # small-signal (linearized) prediction
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.loglog(I * 1e3, true * 1e3, label="true change")
ax.loglog(I * 1e3, superposed * 1e3, "--", label="superposition (wrong)")
ax.loglog(I * 1e3, small * 1e3, ":", label="small-signal: I (R ∥ r_d)")
ax.set_xlabel("added current (mA)"); ax.set_ylabel("change in v_d (mV)"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()
k = np.argmin(abs(I - 1e-5))
print(f"r_d = {rd:.3f} ohm; at 10 uA: true {true[k] * 1e6:.2f} uV, small-signal {small[k] * 1e6:.2f} uV")
assert abs(true[k] - small[k]) / true[k] < 0.01

r_d = 6.002 ohm; at 10 uA: true 59.59 uV, small-signal 59.66 uV


Naive superposition is badly wrong at every level, because the 1 mA alone (with the 5 V source shorted) drives the diode to a completely different operating point. The *small-signal* prediction, a linear model about the 5 V operating point, is accurate for small added currents and drifts away as the current approaches the bias current. That is precisely the domain in which electronics uses superposition.

## Part 3 — DAC linearity from resistor mismatch

**🔮 Predict:** for a 3-bit binary-weighted DAC with random 1% resistor errors, which transition has the worst DNL on average?

In [4]:
def dac(code, R, RL=4.0, Vref=4.0):
    b = [(code >> 2) & 1, (code >> 1) & 1, code & 1]
    G = 1 / np.asarray(R)
    return Vref * (b @ G) / (G.sum() + 1 / RL)

rng = np.random.default_rng(607)
nominal = np.array([1.0, 2.0, 4.0])
worst = np.zeros(7)
for _ in range(1000):
    R = nominal * (1 + rng.uniform(-0.01, 0.01, 3))
    out = np.array([dac(c, R) for c in range(8)])
    lsb = (out[7] - out[0]) / 7                        # end-point LSB
    dnl = np.diff(out) / lsb - 1
    worst = np.maximum(worst, abs(dnl))
for k, w in enumerate(worst):
    print(f"transition {k:03b} -> {k + 1:03b}: worst |DNL| over 1000 DACs = {w:.4f} LSB")
assert int(np.argmax(worst)) == 3                     # the major carry, 011 -> 100

transition 000 -> 001: worst |DNL| over 1000 DACs = 0.0171 LSB
transition 001 -> 010: worst |DNL| over 1000 DACs = 0.0319 LSB
transition 010 -> 011: worst |DNL| over 1000 DACs = 0.0171 LSB
transition 011 -> 100: worst |DNL| over 1000 DACs = 0.0631 LSB
transition 100 -> 101: worst |DNL| over 1000 DACs = 0.0171 LSB
transition 101 -> 110: worst |DNL| over 1000 DACs = 0.0319 LSB
transition 110 -> 111: worst |DNL| over 1000 DACs = 0.0171 LSB


The worst transition is the major carry, 011 → 100, where the MSB switches in and both lower bits switch out, as Example 06-07.5 predicted.

## Part 4 — Worst case against RSS

Three independent error sources at an amplifier's output are each uniformly distributed within ±2, ±3 and ±6 mV.

**🔮 Predict:** the worst case is 11 mV and the RSS of the limits is 7 mV. What fraction of units will exceed 7 mV?

In [5]:
limits = np.array([2.0, 3.0, 6.0])
worst_case = limits.sum()
rss = np.sqrt((limits ** 2).sum())
rng = np.random.default_rng(6070)
total = (rng.uniform(-1, 1, (200_000, 3)) * limits).sum(axis=1)
frac = np.mean(abs(total) > rss)
print(f"worst case {worst_case:.0f} mV, RSS of the limits {rss:.0f} mV")
print(f"Monte Carlo: largest |total| = {abs(total).max():.2f} mV; fraction above the RSS figure = {frac * 100:.2f} %")
print(f"standard deviation of the total = {total.std():.3f} mV (theory {np.sqrt((limits ** 2 / 3).sum()):.3f} mV)")
assert worst_case == 11 and abs(rss - 7) < 1e-12 and 0.05 < frac < 0.10 and abs(total).max() < worst_case
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(total, bins=120, density=True)
for x, lab in ((rss, "RSS"), (worst_case, "worst case")):
    ax.axvline(x, color="0.3", ls="--", lw=0.8); ax.axvline(-x, color="0.3", ls="--", lw=0.8)
    ax.text(x, ax.get_ylim()[1] * 0.9, lab, fontsize=8)
ax.set_xlabel("total error (mV)"); ax.set_ylabel("density")
plt.show()

worst case 11 mV, RSS of the limits 7 mV
Monte Carlo: largest |total| = 10.89 mV; fraction above the RSS figure = 7.40 %
standard deviation of the total = 4.041 mV (theory 4.041 mV)


About 7% of units exceed the 7 mV RSS figure, and none reach the 11 mV worst case (the largest of 200,000 is about 10.9 mV): three errors rarely line up at their extremes. Note what "RSS of the limits" means here. A uniform error with half-width $e$ has standard deviation $e/\sqrt3$, so the total's standard deviation is $7/\sqrt3 = 4.04$ mV, and 7 mV is only about 1.7σ: a "typical" figure, not a bound. If the limits had been ±3σ points of Gaussian errors, their RSS would have been the total's 3σ point instead, exceeded by only 0.27% of units. Always ask what the limits in an RSS budget represent.

## Your turn

1. In Part 1, set the 2 A source of Example 06-07.1 to 3 A. Which source delivers power now, and how much each?
2. In Part 3, replace the resistor errors with a systematic +1% error on the MSB alone. What are the DNL values?
3. In Part 4, make the 6 mV error a ±6 mV *systematic* error that always takes its extreme value in the same direction. How should it be combined with the other two?

<details><summary>Answers</summary>

1. $v_a = 7.2 + 3\times2.4 = 14.4$ V, so the 12 V source now *absorbs* $12\times(14.4 - 12)/4 = 7.2$ W (it is being charged), and the 3 A source delivers $3\times14.4 = 43.2$ W; $R_1$ takes 1.44 W and $R_2$ takes 34.56 W.
2. With the LSB defined from the end points, as in the code, the DNL is −0.034 LSB at the major carry (011 → 100) and +0.006 LSB at each of the other six transitions: the MSB error concentrates at the carry, and the end-point normalization spreads a small compensating error over the rest.
3. Add it directly (worst case) to the RSS of the random ones: $6 + \sqrt{2^2 + 3^2} = 9.61$ mV. Systematic errors do not average out across units.

</details>